In [1]:
!pip install -q streamlit networkx pandas matplotlib

In [2]:
!pip install -q streamlit cloudflared

ERROR: Could not find a version that satisfies the requirement cloudflared (from versions: none)
ERROR: No matching distribution found for cloudflared


In [3]:
import os

print(os.path.exists("/content/peermesh_data.json"))

True


In [4]:
%%writefile peermesh_data.json

{
    "students": [
        {
            "student_id": "S101",
            "faculty_id": "F010",
            "library_id": "L010",
            "placement_id": "P010",
            "name": "Arun Kumar",
            "department": "Computer Science",
            "year": "Final Year",
            "cgpa": 8.4
        },
        {
            "student_id": "S102",
            "faculty_id": "F011",
            "library_id": "L011",
            "placement_id": "P011",
            "name": "Priya Sharma",
            "department": "Artificial Intelligence",
            "year": "Final Year",
            "cgpa": 7.8
        },
        {
            "student_id": "S103",
            "faculty_id": "F012",
            "library_id": "L012",
            "placement_id": "P012",
            "name": "Rahul",
            "department": "Information Technology",
            "year": "Third Year",
            "cgpa": 8.1
        }
    ],

    "faculty": [
        {
            "faculty_id": "F010",
            "student_id": "S101",
            "library_id": "L010",
            "placement_id": "P010",
            "teacher_name": "Dr. Kumar",
            "subject": "Machine Learning",
            "experience": "12 Years",
            "email": "kumar@college.edu"
        },
        {
            "faculty_id": "F011",
            "student_id": "S102",
            "library_id": "L011",
            "placement_id": "P011",
            "teacher_name": "Dr. Priya",
            "subject": "Artificial Intelligence",
            "experience": "10 Years",
            "email": "priya@college.edu"
        },
        {
            "faculty_id": "F012",
            "student_id": "S103",
            "library_id": "L012",
            "placement_id": "P012",
            "teacher_name": "Dr. Raj",
            "subject": "Data Science",
            "experience": "8 Years",
            "email": "raj@college.edu"
        }
    ],

    "library": [
        {
            "library_id": "L010",
            "student_id": "S101",
            "faculty_id": "F010",
            "placement_id": "P010",
            "book_name": "Machine Learning",
            "author": "Tom Mitchell",
            "issue_date": "2026-08-20",
            "fine": 0
        },
        {
            "library_id": "L011",
            "student_id": "S102",
            "faculty_id": "F011",
            "placement_id": "P011",
            "book_name": "Artificial Intelligence",
            "author": "Stuart Russell",
            "issue_date": "2026-08-18",
            "fine": 150
        },
        {
            "library_id": "L012",
            "student_id": "S103",
            "faculty_id": "F012",
            "placement_id": "P012",
            "book_name": "Data Science Handbook",
            "author": "Jake VanderPlas",
            "issue_date": "2026-08-22",
            "fine": 0
        }
    ],

    "placement": [
        {
            "placement_id": "P010",
            "student_id": "S101",
            "faculty_id": "F010",
            "library_id": "L010",
            "company": "TCS",
            "package": "7 LPA",
            "job_role": "ML Engineer",
            "placement_date": "2026-09-15"
        },
        {
            "placement_id": "P011",
            "student_id": "S102",
            "faculty_id": "F011",
            "library_id": "L011",
            "company": "Infosys",
            "package": "6 LPA",
            "job_role": "AI Engineer",
            "placement_date": "2026-09-18"
        },
        {
            "placement_id": "P012",
            "student_id": "S103",
            "faculty_id": "F012",
            "library_id": "L012",
            "company": "Accenture",
            "package": "8 LPA",
            "job_role": "Data Analyst",
            "placement_date": "2026-09-20"
        }
    ]
}

Overwriting peermesh_data.json


In [5]:
%%writefile /content/PeerMesh_updated_app.py

import streamlit as st
import json
import sqlite3
from datetime import datetime
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import os


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="PeerMesh | P2P College System",
    page_icon="🔗",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown("""
<style>

.stApp {
    background-color: #0b0f14;
}

section[data-testid="stSidebar"] {
    background-color: #111827;
}

section[data-testid="stSidebar"] * {
    color: white;
}

.block-container {
    max-width: 1450px;
    padding-top: 1.5rem;
}

h1, h2, h3 {
    color: white !important;
}

p, label {
    color: #e5e7eb !important;
}

div[data-testid="stMetric"] {
    background-color: #111827;
    border: 1px solid #ffffff;
    border-radius: 14px;
    padding: 14px;
}

div[data-testid="stMetric"] label {
    color: #d1d5db !important;
}

div[data-testid="stMetricValue"] {
    color: white !important;
}

.stButton > button {
    border: 1px solid white;
    border-radius: 10px;
    width: 100%;
}

.stTextInput input,
.stTextArea textarea {
    background-color: #111827 !important;
    color: white !important;
    border: 1px solid white !important;
}

</style>
""", unsafe_allow_html=True)


# ============================================================
# DATASET
# ============================================================

DATA_FILE = "/content/peermesh_data.json"


def load_data():

    if not os.path.exists(DATA_FILE):

        st.error(
            "peermesh_data.json was not found in /content."
        )

        st.stop()

    try:

        with open(DATA_FILE, "r", encoding="utf-8") as file:
            return json.load(file)

    except json.JSONDecodeError:

        st.error(
            "peermesh_data.json contains invalid JSON."
        )

        st.stop()


DATA = load_data()


# ============================================================
# AGENT DEFINITIONS
# ============================================================

AGENT_DATA = {

    "Student Agent": {
        "icon": "🎓",
        "role": "Student Information",
        "description":
            "Handles student name, department, year and CGPA.",
        "data":
            DATA.get("students", []),
        "key":
            "student_id"
    },

    "Faculty Agent": {
        "icon": "👨‍🏫",
        "role": "Academic Information",
        "description":
            "Handles faculty, subjects, experience and email.",
        "data":
            DATA.get("faculty", []),
        "key":
            "faculty_id"
    },

    "Library Agent": {
        "icon": "📚",
        "role": "Library Information",
        "description":
            "Handles books, authors, issue dates and fines.",
        "data":
            DATA.get("library", []),
        "key":
            "library_id"
    },

    "Placement Agent": {
        "icon": "💼",
        "role": "Placement Information",
        "description":
            "Handles company, package, job role and placement date.",
        "data":
            DATA.get("placement", []),
        "key":
            "placement_id"
    }
}


AGENT_NAMES = list(AGENT_DATA.keys())


# ============================================================
# NORMALIZATION
# ============================================================

def normalize(value):

    if value is None:
        return ""

    return str(value).strip().upper()


# ============================================================
# FIND RECORD
# ============================================================

def find_record(agent_name, record_id):

    record_id = normalize(record_id)

    key = AGENT_DATA[agent_name]["key"]

    for record in AGENT_DATA[agent_name]["data"]:

        if normalize(record.get(key)) == record_id:

            return record

    return None


# ============================================================
# IDENTIFY AGENT
# ============================================================

def identify_agent(entity_id):

    entity_id = normalize(entity_id)

    if entity_id.startswith("S"):
        return "Student Agent"

    if entity_id.startswith("F"):
        return "Faculty Agent"

    if entity_id.startswith("L"):
        return "Library Agent"

    if entity_id.startswith("P"):
        return "Placement Agent"

    return None


# ============================================================
# GET RECORD
# ============================================================

def get_record(entity_id):

    agent = identify_agent(entity_id)

    if agent is None:
        return None, None

    record = find_record(agent, entity_id)

    return agent, record


# ============================================================
# RELATIONSHIPS
# ============================================================

def faculty_for_student(student_id):

    student = find_record(
        "Student Agent",
        student_id
    )

    if not student:
        return None

    return find_record(
        "Faculty Agent",
        student.get("faculty_id")
    )


def library_for_student(student_id):

    student_id = normalize(student_id)

    return [

        record

        for record in DATA.get("library", [])

        if normalize(
            record.get("student_id")
        ) == student_id

    ]


def placement_for_student(student_id):

    student_id = normalize(student_id)

    return [

        record

        for record in DATA.get("placement", [])

        if normalize(
            record.get("student_id")
        ) == student_id

    ]


def student_for_library(library_id):

    library = find_record(
        "Library Agent",
        library_id
    )

    if not library:
        return None

    return find_record(
        "Student Agent",
        library.get("student_id")
    )


def student_for_placement(placement_id):

    placement = find_record(
        "Placement Agent",
        placement_id
    )

    if not placement:
        return None

    return find_record(
        "Student Agent",
        placement.get("student_id")
    )


def students_for_faculty(faculty_id):

    faculty_id = normalize(faculty_id)

    return [

        record

        for record in DATA.get("students", [])

        if normalize(
            record.get("faculty_id")
        ) == faculty_id

    ]


# ============================================================
# P2P NETWORK
# ============================================================

NETWORK = nx.Graph()

NETWORK.add_nodes_from(AGENT_NAMES)

for i in range(len(AGENT_NAMES)):

    for j in range(i + 1, len(AGENT_NAMES)):

        NETWORK.add_edge(
            AGENT_NAMES[i],
            AGENT_NAMES[j]
        )


# ============================================================
# SQLITE DATABASE
# ============================================================

DB_FILE = "/content/peermesh_communication.db"


def get_connection():

    return sqlite3.connect(DB_FILE)


def initialize_database():

    connection = get_connection()

    connection.execute("""
        CREATE TABLE IF NOT EXISTS communication_history (

            id INTEGER PRIMARY KEY AUTOINCREMENT,

            sender TEXT,

            receiver TEXT,

            message_type TEXT,

            content TEXT,

            priority TEXT,

            timestamp TEXT,

            status TEXT

        )
    """)

    connection.commit()

    connection.close()


initialize_database()


# ============================================================
# LOG COMMUNICATION
# ============================================================

def log_communication(
    sender,
    receiver,
    message_type,
    content,
    priority="NORMAL",
    status="DELIVERED"
):

    connection = get_connection()

    connection.execute("""
        INSERT INTO communication_history
        (
            sender,
            receiver,
            message_type,
            content,
            priority,
            timestamp,
            status
        )

        VALUES (?, ?, ?, ?, ?, ?, ?)

    """, (

        sender,
        receiver,
        message_type,
        content,
        priority,
        datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
        status

    ))

    connection.commit()

    connection.close()


# ============================================================
# GET COMMUNICATION HISTORY
# ============================================================

def get_history():

    connection = get_connection()

    dataframe = pd.read_sql_query("""

        SELECT

            sender AS Sender,

            receiver AS Receiver,

            message_type AS Type,

            content AS Content,

            priority AS Priority,

            timestamp AS Timestamp,

            status AS Status

        FROM communication_history

        ORDER BY id DESC

    """, connection)

    connection.close()

    return dataframe


# ============================================================
# DIRECT P2P COMMUNICATION
# ============================================================

def direct_p2p(
    sender,
    receiver,
    message,
    message_type="REQUEST",
    priority="NORMAL"
):

    if sender == receiver:

        return False, "Sender and receiver must be different."


    if not NETWORK.has_edge(
        sender,
        receiver
    ):

        return False, "No direct P2P connection."


    log_communication(
        sender,
        receiver,
        message_type,
        message,
        priority,
        "DELIVERED"
    )


    return True, "Message delivered."


# ============================================================
# QUESTION FIELD DETECTION
# ============================================================

def detect_requested_field(question):

    q = question.lower().strip()


    # CGPA MUST COME FIRST
    if (
        "cgpa" in q
        or "c.g.p.a" in q
        or "grade point" in q
    ):

        return "cgpa"


    if (
        "department" in q
        or "dept" in q
        or "branch" in q
    ):

        return "department"


    if (
        "student name" in q
        or "my name" in q
        or "name" in q
        or "who am i" in q
    ):

        return "name"


    if "year" in q:

        return "year"


    if (
        "faculty" in q
        or "teacher" in q
        or "professor" in q
    ):

        return "faculty"


    if (
        "subject" in q
    ):

        return "subject"


    if (
        "book" in q
        or "library" in q
        or "borrow" in q
    ):

        return "book"


    if "author" in q:

        return "author"


    if "fine" in q:

        return "fine"


    if (
        "company" in q
        or "placement" in q
        or "selected" in q
    ):

        return "company"


    if "package" in q:

        return "package"


    if (
        "job role" in q
        or "job" in q
        or "role" in q
    ):

        return "job_role"


    if (
        "placement date" in q
        or "joining date" in q
    ):

        return "placement_date"


    if (
        "experience" in q
    ):

        return "experience"


    if "email" in q:

        return "email"


    if (
        "complete" in q
        or "everything" in q
        or "all information" in q
    ):

        return "complete"


    return "general"


# ============================================================
# STUDENT ANSWERS
# ============================================================

def student_answer(
    student_id,
    question
):

    student = find_record(
        "Student Agent",
        student_id
    )

    field = detect_requested_field(question)


    if not student:

        return {
            "success": False,
            "answer": "Student not found.",
            "targets": []
        }


    if field == "cgpa":

        return {
            "success": True,
            "answer":
                f"CGPA: {student.get('cgpa', 'Not Available')}",
            "targets": ["Faculty Agent"]
        }


    if field == "name":

        return {
            "success": True,
            "answer":
                f"Name: {student.get('name', 'Not Available')}",
            "targets": []
        }


    if field == "department":

        return {
            "success": True,
            "answer":
                f"Department: {student.get('department', 'Not Available')}",
            "targets": []
        }


    if field == "year":

        return {
            "success": True,
            "answer":
                f"Year: {student.get('year', 'Not Available')}",
            "targets": []
        }


    if field == "faculty":

        faculty = faculty_for_student(
            student_id
        )

        if not faculty:

            return {
                "success": False,
                "answer": "Faculty information not found.",
                "targets": ["Faculty Agent"]
            }

        return {
            "success": True,

            "answer":
                f"Faculty: {faculty.get('teacher_name')}\n"
                f"Subject: {faculty.get('subject')}",

            "targets": ["Faculty Agent"]
        }


    if field == "book":

        books = library_for_student(
            student_id
        )

        if not books:

            return {
                "success": False,
                "answer": "No library record found.",
                "targets": ["Library Agent"]
            }

        book = books[0]

        return {
            "success": True,

            "answer":
                f"Book: {book.get('book_name')}",

            "targets": ["Library Agent"]
        }


    if field == "author":

        books = library_for_student(
            student_id
        )

        if not books:

            return {
                "success": False,
                "answer": "No library record found.",
                "targets": ["Library Agent"]
            }

        return {
            "success": True,

            "answer":
                f"Author: {books[0].get('author')}",

            "targets": ["Library Agent"]
        }


    if field == "fine":

        books = library_for_student(
            student_id
        )

        total_fine = sum(

            float(
                book.get("fine", 0) or 0
            )

            for book in books

        )

        return {
            "success": True,

            "answer":
                f"Fine: ₹{total_fine:g}",

            "targets": ["Library Agent"]
        }


    if field == "company":

        placements = placement_for_student(
            student_id
        )

        if not placements:

            return {
                "success": False,
                "answer": "No placement record found.",
                "targets": ["Placement Agent"]
            }

        return {
            "success": True,

            "answer":
                f"Company: {placements[0].get('company')}",

            "targets": ["Placement Agent"]
        }


    if field == "package":

        placements = placement_for_student(
            student_id
        )

        if not placements:

            return {
                "success": False,
                "answer": "No placement record found.",
                "targets": ["Placement Agent"]
            }

        return {
            "success": True,

            "answer":
                f"Package: {placements[0].get('package')}",

            "targets": ["Placement Agent"]
        }


    if field == "job_role":

        placements = placement_for_student(
            student_id
        )

        if not placements:

            return {
                "success": False,
                "answer": "No placement record found.",
                "targets": ["Placement Agent"]
            }

        return {
            "success": True,

            "answer":
                f"Job Role: {placements[0].get('job_role')}",

            "targets": ["Placement Agent"]
        }


    if field == "placement_date":

        placements = placement_for_student(
            student_id
        )

        if not placements:

            return {
                "success": False,
                "answer": "No placement record found.",
                "targets": ["Placement Agent"]
            }

        return {
            "success": True,

            "answer":
                f"Placement Date: {placements[0].get('placement_date')}",

            "targets": ["Placement Agent"]
        }


    if field == "complete":

        faculty = faculty_for_student(
            student_id
        )

        library = library_for_student(
            student_id
        )

        placement = placement_for_student(
            student_id
        )

        return {
            "success": True,

            "answer": "Complete information retrieved.",

            "targets": [
                "Faculty Agent",
                "Library Agent",
                "Placement Agent"
            ],

            "complete": True,

            "student": student,

            "faculty": faculty,

            "library": library,

            "placement": placement
        }


    return {
        "success": False,

        "answer":
            "Please ask for a specific field such as CGPA, name, department, faculty, book, company or package.",

        "targets": []
    }


# ============================================================
# FACULTY ANSWER
# ============================================================

def faculty_answer(
    faculty_id,
    question
):

    faculty = find_record(
        "Faculty Agent",
        faculty_id
    )

    if not faculty:

        return {
            "success": False,
            "answer": "Faculty not found.",
            "targets": []
        }


    q = question.lower()


    if "experience" in q:

        return {
            "success": True,
            "answer":
                f"Experience: {faculty.get('experience')}",
            "targets": []
        }


    if "email" in q:

        return {
            "success": True,
            "answer":
                f"Email: {faculty.get('email')}",
            "targets": []
        }


    if "subject" in q:

        return {
            "success": True,
            "answer":
                f"Subject: {faculty.get('subject')}",
            "targets": []
        }


    if "student" in q:

        students = students_for_faculty(
            faculty_id
        )

        if students:

            names = "\n".join(

                f"{x.get('student_id')}: {x.get('name')}"

                for x in students

            )

            return {
                "success": True,
                "answer": f"Students:\n{names}",
                "targets": ["Student Agent"]
            }


    return {
        "success": True,

        "answer":
            f"Faculty: {faculty.get('teacher_name')}",

        "targets": []
    }


# ============================================================
# LIBRARY ANSWER
# ============================================================

def library_answer(
    library_id,
    question
):

    library = find_record(
        "Library Agent",
        library_id
    )

    if not library:

        return {
            "success": False,
            "answer": "Library record not found.",
            "targets": []
        }


    q = question.lower()


    if "author" in q:

        answer = (
            f"Author: "
            f"{library.get('author', 'Not Available')}"
        )


    elif "fine" in q:

        answer = (
            f"Fine: ₹"
            f"{float(library.get('fine', 0) or 0):g}"
        )


    elif (
        "issue date" in q
        or "issued" in q
    ):

        answer = (
            f"Issue Date: "
            f"{library.get('issue_date', 'Not Available')}"
        )


    elif "student" in q:

        student = student_for_library(
            library_id
        )

        if student:

            answer = (
                f"Student: "
                f"{student.get('name')} "
                f"({student.get('student_id')})"
            )

        else:

            answer = "Student information not found."


    else:

        answer = (
            f"Book: "
            f"{library.get('book_name', 'Not Available')}"
        )


    targets = []

    if "student" in q:
        targets = ["Student Agent"]


    return {
        "success": True,
        "answer": answer,
        "targets": targets
    }


# ============================================================
# PLACEMENT ANSWER
# ============================================================

def placement_answer(
    placement_id,
    question
):

    placement = find_record(
        "Placement Agent",
        placement_id
    )

    if not placement:

        return {
            "success": False,
            "answer": "Placement record not found.",
            "targets": []
        }


    q = question.lower()


    if "company" in q:

        answer = (
            f"Company: "
            f"{placement.get('company')}"
        )


    elif "package" in q:

        answer = (
            f"Package: "
            f"{placement.get('package')}"
        )


    elif (
        "job role" in q
        or "role" in q
        or "job" in q
    ):

        answer = (
            f"Job Role: "
            f"{placement.get('job_role')}"
        )


    elif "date" in q:

        answer = (
            f"Placement Date: "
            f"{placement.get('placement_date')}"
        )


    elif "student" in q:

        student = student_for_placement(
            placement_id
        )

        if student:

            answer = (
                f"Student: "
                f"{student.get('name')} "
                f"({student.get('student_id')})"
            )

        else:

            answer = "Student information not found."


    else:

        answer = (
            f"Company: "
            f"{placement.get('company')}\n"
            f"Package: "
            f"{placement.get('package')}\n"
            f"Job Role: "
            f"{placement.get('job_role')}"
        )


    targets = []

    if "student" in q:

        targets = ["Student Agent"]


    return {
        "success": True,
        "answer": answer,
        "targets": targets
    }


# ============================================================
# MAIN QUESTION PROCESSOR
# ============================================================

def process_question(
    entity_id,
    question
):

    agent, record = get_record(
        entity_id
    )


    if not record:

        return {
            "success": False,
            "answer":
                f"ID {entity_id} was not found.",
            "targets": []
        }


    if agent == "Student Agent":

        return student_answer(
            entity_id,
            question
        )


    if agent == "Faculty Agent":

        return faculty_answer(
            entity_id,
            question
        )


    if agent == "Library Agent":

        return library_answer(
            entity_id,
            question
        )


    if agent == "Placement Agent":

        return placement_answer(
            entity_id,
            question
        )


    return {
        "success": False,
        "answer": "Unable to process the question.",
        "targets": []
    }


# ============================================================
# DISPLAY ANSWER
# ONLY CHANGE MADE HERE
# ============================================================

def display_answer(result):
    """
    Display only one final answer.
    No complete record or JSON is displayed.
    """

    if not result:
        st.error("No answer received.")
        return

    answer = result.get(
        "answer",
        "No answer available."
    )

    if result.get("success"):

        st.write(answer)

    else:

        st.warning(answer)


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.title("🔗 PeerMesh")

    st.caption(
        "P2P Multi-Agent College Data Sharing System"
    )

    st.divider()


    menu = [

        "🏠 Dashboard",

        "🧠 Ask Agent Network",

        "📡 Direct P2P",

        "🎓 Student Profiles",

        "👨‍🏫 Faculty Agent",

        "📚 Library Agent",

        "💼 Placement Agent",

        "🌐 P2P Network",

        "📜 Communication History"

    ]


    choice = st.radio(
        "Navigation",
        menu
    )


    st.divider()

    st.success(
        "All Agents Online"
    )

    st.caption(
        "Python + Streamlit + NetworkX + SQLite"
    )


# ============================================================
# DASHBOARD
# ============================================================

if choice == "🏠 Dashboard":

    st.title("🔗 PeerMesh")

    st.subheader(
        "P2P Multi-Agent College Data Sharing System"
    )


    st.write(
        "A fully connected Peer-to-Peer network "
        "where Student, Faculty, Library and Placement "
        "agents communicate directly."
    )


    c1, c2, c3, c4 = st.columns(4)


    with c1:

        st.metric(
            "Active Agents",
            len(AGENT_NAMES)
        )


    with c2:

        st.metric(
            "Students",
            len(DATA.get("students", []))
        )


    with c3:

        st.metric(
            "P2P Connections",
            NETWORK.number_of_edges()
        )


    with c4:

        st.metric(
            "Communication Directions",
            NETWORK.number_of_edges() * 2
        )


    st.subheader(
        "🤖 Agent Network"
    )


    columns = st.columns(4)


    for column, (name, info) in zip(
        columns,
        AGENT_DATA.items()
    ):

        with column:

            st.markdown(
                f"### {info['icon']} {name}"
            )

            st.write(
                f"**Role:** {info['role']}"
            )

            st.write(
                info["description"]
            )

            st.success(
                "ONLINE"
            )


    st.subheader(
        "🔄 How PeerMesh Works"
    )


    st.write(
        "1. User enters an Entity ID."
    )

    st.write(
        "2. User asks a specific question."
    )

    st.write(
        "3. The correct agent identifies the requested field."
    )

    st.write(
        "4. Required information is obtained directly or through a peer agent."
    )

    st.write(
        "5. Only the requested answer is displayed."
    )


# ============================================================
# ASK AGENT NETWORK
# ============================================================

elif choice == "🧠 Ask Agent Network":

    st.title(
        "🧠 Ask Agent Network"
    )


    st.write(
        "Ask a specific question using your Entity ID."
    )


    entity_id = st.text_input(
        "Entity ID",
        placeholder=
            "S101 / S102 / S103 / F010 / L010 / P010"
    )


    entity_id = normalize(
        entity_id
    )


    question = st.text_area(
        "Your Question",
        placeholder=
            "What is my CGPA?",
        height=120
    )


    if st.button(
        "🚀 Send Query",
        use_container_width=True
    ):


        if not entity_id:

            st.warning(
                "Please enter an Entity ID."
            )

            st.stop()


        if not question.strip():

            st.warning(
                "Please enter your question."
            )

            st.stop()


        agent, record = get_record(
            entity_id
        )


        if not record:

            st.error(
                f"ID {entity_id} does not exist."
            )

            st.stop()


        st.info(
            f"Request started from {agent}: {entity_id}"
        )


        result = process_question(
            entity_id,
            question
        )


        # ----------------------------------------------------
        # P2P COMMUNICATION
        # ----------------------------------------------------

        if result.get("targets"):

            st.subheader(
                "📡 P2P Agent Communication"
            )


            for target in result["targets"]:

                success, message = direct_p2p(

                    agent,

                    target,

                    f"Request: {question.strip()}",

                    "REQUEST",

                    "HIGH"

                )


                if success:

                    st.success(
                        f"🔗 {agent} → {target}"
                    )

                else:

                    st.error(
                        message
                    )


        # ----------------------------------------------------
        # ANSWER
        # ----------------------------------------------------

        st.subheader(
            "📥 Answer"
        )


        display_answer(
            result
        )


# ============================================================
# DIRECT P2P
# ============================================================

elif choice == "📡 Direct P2P":

    st.title(
        "📡 Direct P2P Communication"
    )


    st.write(
        "Every agent has a direct connection with every other agent."
    )


    c1, c2 = st.columns(2)


    with c1:

        sender = st.selectbox(
            "Sender Agent",
            AGENT_NAMES
        )


    with c2:

        receiver_options = [

            agent

            for agent in AGENT_NAMES

            if agent != sender

        ]


        receiver = st.selectbox(
            "Receiver Agent",
            receiver_options
        )


    entity_id = st.text_input(
        "Entity ID",
        placeholder="S101 / F010 / L010 / P010"
    )


    message = st.text_area(
        "Message",
        placeholder="Request information from peer",
        height=100
    )


    message_type = st.selectbox(
        "Message Type",
        [
            "REQUEST",
            "RESPONSE",
            "QUERY",
            "DATA"
        ]
    )


    priority = st.selectbox(
        "Priority",
        [
            "LOW",
            "NORMAL",
            "HIGH",
            "CRITICAL"
        ]
    )


    if st.button(
        "📨 Send P2P Request",
        use_container_width=True
    ):

        if not entity_id.strip():

            st.warning(
                "Please enter an Entity ID."
            )

        elif not message.strip():

            st.warning(
                "Please enter a message."
            )

        else:

            success, status = direct_p2p(

                sender,

                receiver,

                message,

                message_type,

                priority

            )


            if success:

                st.success(
                    f"🔗 {sender} → {receiver}"
                )

                st.write(
                    "Message delivered through direct P2P connection."
                )

                # Get the actual answer for the entered Entity ID
                result = process_question(
                    entity_id.strip(),
                    message.strip()
                )

                st.subheader(
                    "📥 Response"
                )

                display_answer(
                    result
                )

            else:

                st.error(
                    status
                )


# ============================================================
# STUDENT PROFILES
# ============================================================

elif choice == "🎓 Student Profiles":

    st.title(
        "🎓 Student Profiles"
    )


    students = DATA.get(
        "students",
        []
    )


    st.dataframe(
        pd.DataFrame(students),
        use_container_width=True,
        hide_index=True
    )


    if students:

        student_ids = [

            student["student_id"]

            for student in students

        ]


        selected_id = st.selectbox(
            "Select Student",
            student_ids
        )


        student = find_record(
            "Student Agent",
            selected_id
        )


        c1, c2, c3, c4 = st.columns(4)


        with c1:

            st.metric(
                "Name",
                student.get(
                    "name",
                    "-"
                )
            )


        with c2:

            st.metric(
                "Department",
                student.get(
                    "department",
                    "-"
                )
            )


        with c3:

            st.metric(
                "Year",
                student.get(
                    "year",
                    "-"
                )
            )


        with c4:

            st.metric(
                "CGPA",
                student.get(
                    "cgpa",
                    "-"
                )
            )


# ============================================================
# FACULTY
# ============================================================

elif choice == "👨‍🏫 Faculty Agent":

    st.title(
        "👨‍🏫 Faculty Agent"
    )


    st.dataframe(
        pd.DataFrame(
            DATA.get(
                "faculty",
                []
            )
        ),
        use_container_width=True,
        hide_index=True
    )


# ============================================================
# LIBRARY
# ============================================================

elif choice == "📚 Library Agent":

    st.title(
        "📚 Library Agent"
    )


    library = DATA.get(
        "library",
        []
    )


    st.dataframe(
        pd.DataFrame(library),
        use_container_width=True,
        hide_index=True
    )


    total_fine = sum(

        float(
            record.get(
                "fine",
                0
            ) or 0
        )

        for record in library

    )


    students_with_fine = sum(

        1

        for record in library

        if float(
            record.get(
                "fine",
                0
            ) or 0
        ) > 0

    )


    c1, c2 = st.columns(2)


    with c1:

        st.metric(
            "Total Outstanding Fine",
            f"₹{total_fine:g}"
        )


    with c2:

        st.metric(
            "Students With Fine",
            students_with_fine
        )


# ============================================================
# PLACEMENT
# ============================================================

elif choice == "💼 Placement Agent":

    st.title(
        "💼 Placement Agent"
    )


    placement = DATA.get(
        "placement",
        []
    )


    st.dataframe(
        pd.DataFrame(placement),
        use_container_width=True,
        hide_index=True
    )


    packages = []


    for record in placement:

        try:

            value = float(

                str(
                    record.get(
                        "package",
                        ""
                    )
                )
                .replace(
                    "LPA",
                    ""
                )
                .strip()

            )

            packages.append(value)

        except ValueError:

            pass


    companies = {

        record.get("company")

        for record in placement

        if record.get("company")

    }


    c1, c2, c3 = st.columns(3)


    with c1:

        st.metric(
            "Placement Records",
            len(placement)
        )


    with c2:

        st.metric(
            "Companies",
            len(companies)
        )


    with c3:

        st.metric(
            "Highest Package",
            f"{max(packages):g} LPA"
            if packages
            else "N/A"
        )


# ============================================================
# P2P NETWORK
# ============================================================

elif choice == "🌐 P2P Network":

    st.title(
        "🌐 Fully Connected P2P Network"
    )


    st.write(
        "Every agent has a direct connection with every other agent."
    )


    c1, c2, c3 = st.columns(3)


    with c1:

        st.metric(
            "Agents",
            NETWORK.number_of_nodes()
        )


    with c2:

        st.metric(
            "P2P Connections",
            NETWORK.number_of_edges()
        )


    with c3:

        st.metric(
            "Communication Directions",
            NETWORK.number_of_edges() * 2
        )


    figure, axis = plt.subplots(
        figsize=(10, 7)
    )


    positions = nx.spring_layout(
        NETWORK,
        seed=42
    )


    nx.draw_networkx(

        NETWORK,

        positions,

        ax=axis,

        with_labels=True,

        node_size=4500,

        font_size=9,

        font_weight="bold"

    )


    axis.set_title(
        "PeerMesh Multi-Agent P2P Network"
    )

    axis.axis("off")


    st.pyplot(
        figure
    )


    plt.close(
        figure
    )


    st.subheader(
        "🔗 Direct Connections"
    )


    connections = pd.DataFrame(

        list(
            NETWORK.edges()
        ),

        columns=[
            "Agent A",
            "Agent B"
        ]

    )


    st.dataframe(
        connections,
        use_container_width=True,
        hide_index=True
    )


# ============================================================
# COMMUNICATION HISTORY
# ============================================================

elif choice == "📜 Communication History":

    st.title(
        "📜 Communication History"
    )


    c1, c2 = st.columns(2)


    with c1:

        if st.button(
            "🔄 Refresh",
            use_container_width=True
        ):

            st.rerun()


    with c2:

        if st.button(
            "🗑️ Clear History",
            use_container_width=True
        ):

            connection = get_connection()

            connection.execute(
                "DELETE FROM communication_history"
            )

            connection.commit()

            connection.close()

            st.success(
                "Communication history cleared."
            )

            st.rerun()


    dataframe = get_history()


    if dataframe.empty:

        st.info(
            "No communication has been recorded yet."
        )

    else:

        st.dataframe(
            dataframe,
            use_container_width=True,
            hide_index=True
        )

Overwriting /content/PeerMesh_updated_app.py


In [6]:
!pkill -f streamlit

In [7]:
!streamlit run /content/PeerMesh_updated_app.py > streamlit.log 2>&1 &

In [8]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [9]:
!streamlit run /content/PeerMesh_updated_app.py --server.port 8501 &>/content/streamlit.log &

In [ ]:
!./cloudflared tunnel --url http://localhost:8501

2026-09-03T06:35:35Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-03T06:35:35Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-03T06:35:39Z INF +--------------------------------------------------------------------------------------------+
2026-09-03T06:35:39Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-03T06:35:39Z INF |  https://wallace-suggestions-acrylic-strength.trycloud